# 03 — Classical ML Baseline

ECFP fingerprints + RDKit descriptors -> XGBoost / LightGBM / Random Forest,
evaluated on both the scaffold split (primary) and random split (contrast only).
Uncertainty via conformal prediction intervals (MAPIE). Applicability domain
via Tanimoto similarity to nearest training neighbor.

In [1]:
import sys
from pathlib import Path

import joblib
import mlflow
import pandas as pd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.components.feature_engineering import featurize
from src.components.model_evaluator import applicability_domain, regression_metrics
from src.components.model_trainer import predict_with_interval, train_with_uncertainty

config = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
SEED = config["seed"]

mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"].replace("sqlite:///", f"sqlite:///{ROOT}/"))
EXPERIMENT = config["mlflow"]["experiment_name"]
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(EXPERIMENT, artifact_location=str(ROOT / "mlruns" / "artifacts"))
mlflow.set_experiment(EXPERIMENT)

2026/07/22 23:05:18 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/22 23:05:18 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='/Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/mlruns/artifacts', creation_time=1784754319542, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784754319542, lifecycle_stage='active', name='egfr_pic50_classical_ml', tags={}, trace_location=None, workspace='default'>

In [2]:
SPLITS_DIR = ROOT / config["data"]["splits_dir"]

def load_split(name, part):
    return pd.read_csv(SPLITS_DIR / f"{name}_{part}.csv")

splits = {
    name: {part: load_split(name, part) for part in ("train", "val", "test")}
    for name in ("scaffold", "random")
}
{name: {p: len(df) for p, df in parts.items()} for name, parts in splits.items()}

{'scaffold': {'train': 8401, 'val': 1050, 'test': 1051},
 'random': {'train': 8401, 'val': 1050, 'test': 1051}}

## Train grid

Model type x split type, ECFP fingerprints (the standard classical-ML baseline
representation). Each run is logged to MLflow with split type, descriptor
type, hyperparameters, seed, dataset size, and val/test metrics.

In [3]:
MODEL_TYPES = ["xgboost", "lightgbm", "random_forest"]
DESCRIPTOR_KIND = "ecfp"
N_BITS = config["features"]["ecfp_n_bits"]
AD_THRESHOLD = config["applicability_domain"]["tanimoto_threshold"]

results = []
trained_models = {}

for split_name, parts in splits.items():
    Xtr = featurize(parts["train"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
    Xv = featurize(parts["val"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
    Xte = featurize(parts["test"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
    ytr, yv, yte = (parts[p]["pIC50"].values for p in ("train", "val", "test"))

    for model_type in MODEL_TYPES:
        with mlflow.start_run(run_name=f"{model_type}_{split_name}"):
            model = train_with_uncertainty(model_type, Xtr, ytr, Xv, yv, seed=SEED)

            val_pred, val_lo, val_hi = predict_with_interval(model, Xv)
            test_pred, test_lo, test_hi = predict_with_interval(model, Xte)
            val_metrics = regression_metrics(yv, val_pred)
            test_metrics = regression_metrics(yte, test_pred)

            sim, in_domain = applicability_domain(
                parts["test"]["smiles"], parts["train"]["smiles"], threshold=AD_THRESHOLD
            )

            mlflow.log_params({
                "model_type": model_type,
                "split_type": split_name,
                "descriptor_type": DESCRIPTOR_KIND,
                "seed": SEED,
                "n_train": len(Xtr),
                "n_val": len(Xv),
                "n_test": len(Xte),
            })
            mlflow.log_metrics({f"val_{k}": v for k, v in val_metrics.items()})
            mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()})
            mlflow.log_metric("test_mean_interval_width", (test_hi - test_lo).mean())
            mlflow.log_metric("test_ad_in_domain_frac", in_domain.mean())
            mlflow.sklearn.log_model(model, name="model", input_example=Xtr[:2])

            trained_models[(model_type, split_name)] = model
            results.append({
                "model_type": model_type,
                "split_type": split_name,
                **{f"val_{k}": v for k, v in val_metrics.items()},
                **{f"test_{k}": v for k, v in test_metrics.items()},
                "test_mean_interval_width": (test_hi - test_lo).mean(),
                "test_ad_in_domain_frac": in_domain.mean(),
            })

results_df = pd.DataFrame(results)
results_df

2026/07/22 23:05:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


2026/07/22 23:05:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


...

2026/07/22 23:08:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


2026/07/22 23:11:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,model_type,split_type,val_rmse,val_mae,val_r2,val_spearman,test_rmse,test_mae,test_r2,test_spearman,test_mean_interval_width,test_ad_in_domain_frac
0,xgboost,scaffold,0.986609,0.736647,0.433478,0.692492,0.947434,0.714775,0.451204,0.690528,3.106117,0.900095
1,lightgbm,scaffold,0.954630,0.725477,0.469610,0.698393,0.934020,0.710831,0.466634,0.699646,3.066953,0.900095
2,random_forest,scaffold,0.932637,0.704575,0.493766,0.717464,0.889454,0.665822,0.516317,0.732900,3.023000,0.900095
3,xgboost,random,0.784003,0.584638,0.670125,0.831161,0.803522,0.601349,0.668117,0.821009,2.444267,0.976213
4,lightgbm,random,0.795806,0.607411,0.660118,0.820061,0.817217,0.618731,0.656708,0.818824,2.670097,0.976213
5,random_forest,random,0.714712,0.514834,0.725858,0.856938,0.764138,0.543703,0.699854,0.837687,2.307700,0.976213


## Results

Scaffold split is the headline number per project methodology — random split
is shown only for contrast (it typically looks better because train/test
share scaffolds, which inflates apparent generalization).

In [4]:
results_df.sort_values(["split_type", "test_rmse"]).reset_index(drop=True)

,model_type,split_type,val_rmse,val_mae,val_r2,val_spearman,test_rmse,test_mae,test_r2,test_spearman,test_mean_interval_width,test_ad_in_domain_frac
0,random_forest,random,0.714712,0.514834,0.725858,0.856938,0.764138,0.543703,0.699854,0.837687,2.307700,0.976213
1,xgboost,random,0.784003,0.584638,0.670125,0.831161,0.803522,0.601349,0.668117,0.821009,2.444267,0.976213
2,lightgbm,random,0.795806,0.607411,0.660118,0.820061,0.817217,0.618731,0.656708,0.818824,2.670097,0.976213
3,random_forest,scaffold,0.932637,0.704575,0.493766,0.717464,0.889454,0.665822,0.516317,0.732900,3.023000,0.900095
4,lightgbm,scaffold,0.954630,0.725477,0.469610,0.698393,0.934020,0.710831,0.466634,0.699646,3.066953,0.900095
5,xgboost,scaffold,0.986609,0.736647,0.433478,0.692492,0.947434,0.714775,0.451204,0.690528,3.106117,0.900095


In [5]:
(ROOT / config["results_dir"]).mkdir(exist_ok=True)
results_df.to_csv(ROOT / config["results_dir"] / "classical_ml_results.csv", index=False)

gap = (
    results_df.pivot(index="model_type", columns="split_type", values="test_rmse")
    .assign(scaffold_minus_random=lambda d: d["scaffold"] - d["random"])
)
gap

split_type,random,scaffold,scaffold_minus_random
model_type,,,
lightgbm,0.817217,0.934020,0.116802
random_forest,0.764138,0.889454,0.125316
xgboost,0.803522,0.947434,0.143911


## Register best model per model class

Best = lowest scaffold-split val RMSE (scaffold is the primary evaluation
split, so model selection should use it — not the optimistic random split).

In [6]:
(ROOT / config["models_dir"]).mkdir(exist_ok=True)

best_per_class = (
    results_df[results_df["split_type"] == "scaffold"]
    .sort_values("val_rmse")
    .groupby("model_type")
    .first()
)

for model_type, row in best_per_class.iterrows():
    model = trained_models[(model_type, "scaffold")]
    path = ROOT / config["models_dir"] / f"{model_type}_scaffold.joblib"
    joblib.dump(model, path)

    with mlflow.start_run(run_name=f"register_{model_type}"):
        mlflow.log_params({"model_type": model_type, "split_type": "scaffold", "seed": SEED})
        mlflow.log_metrics({f"test_{k}": row[f"test_{k}"] for k in ("rmse", "mae", "r2", "spearman")})
        mlflow.sklearn.log_model(
            model, name="model", registered_model_name=f"egfr_pic50_{model_type}"
        )

best_per_class

2026/07/22 23:11:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'egfr_pic50_lightgbm'.
Created version '1' of model 'egfr_pic50_lightgbm'.
2026/07/22 23:11:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'egfr_pic50_random_forest'.
Created version '1' of model 'egfr_pic50_random_forest'.
2026/07/22 23:11:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Successfully registered model 'egfr_pic50_xgboost'.
Created version '1' of model 'egfr_pic50_xgboost'.


,split_type,val_rmse,val_mae,val_r2,val_spearman,test_rmse,test_mae,test_r2,test_spearman,test_mean_interval_width,test_ad_in_domain_frac
model_type,,,,,,,,,,,
lightgbm,scaffold,0.954630,0.725477,0.469610,0.698393,0.934020,0.710831,0.466634,0.699646,3.066953,0.900095
random_forest,scaffold,0.932637,0.704575,0.493766,0.717464,0.889454,0.665822,0.516317,0.732900,3.023000,0.900095
xgboost,scaffold,0.986609,0.736647,0.433478,0.692492,0.947434,0.714775,0.451204,0.690528,3.106117,0.900095
